# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itz-me-sree/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Can observable content and search-performance signals be used to identify pages that are likely to need a content refresh and rank those pages by refresh opportunity?

### Decision Supported

The analysis supports the decision of which pages should be reviewed and potentially refreshed first, helping content teams prioritize limited review and optimization effort instead of manually reviewing every page.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

The analysis uses the anonymized FlyRank content-performance dataset provided
for the internship.

The dataset contains 30,000 content records and 44 columns. Each row represents
one content item with information about its search demand, content
characteristics, historical performance, engagement, freshness, and observed
performance trend.

### Main Feature Groups

The analysis considers:

- Search-demand signals such as search volume, competition, and CPC
- Content characteristics such as word count and character count
- Historical performance such as impressions, clicks, sessions, users, and pageviews
- Recent and previous-period performance
- Engagement signals such as engagement rate and scroll rate
- Freshness signals such as content age and days since last update

### Target

The dataset does not contain a direct human-labelled "needs refresh" outcome.
Therefore, observed performance decline is used as a proxy for refresh
opportunity.

A page is assigned:

- `1` when `trend_direction` is `down`
- `0` for other observed trend categories

This proxy represents observed performance decline and should not be interpreted
as a confirmed business decision that a page must be refreshed.

### Leakage Control

`trend_direction` and `trend_pct` are excluded from the model features because
they directly describe the observed performance trend used to construct the
target.

Identifiers and derived categorical fields are also excluded from the baseline
feature vector.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology

The analysis uses a supervised binary classification approach to identify
pages associated with observed performance decline.

The target is a refresh-opportunity proxy derived from `trend_direction`.
A value of 1 represents observed performance decline, while 0 represents
other observed trend categories.

The workflow first establishes a transparent baseline rule and then compares
Logistic Regression, Decision Tree, and Random Forest models. The final model
is selected using Precision@50 because the practical goal is to rank a small
number of pages for human review.

The feature vector includes search-demand, content, historical performance,
engagement, and freshness signals. Known leakage-prone fields
`trend_direction` and `trend_pct` are excluded from model features because
they are used to construct the target.

The final evaluation uses a client-level holdout split so that pages from
held-out clients are not used for model training. Model probabilities are
then used to rank pages by refresh opportunity.

The resulting ranking is intended for decision support and human review,
rather than automatic refresh decisions.

In [5]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [6]:
import os

print("Current folder:", os.getcwd())
print(
    "CSV exists:",
    os.path.exists("data/raw/content_refresh_anonymized.csv")
)

Current folder: /content/flyrank-ml-internship
CSV exists: True


In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

Dataset shape: (30000, 44)
Columns: 44


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score

# Target: observed decline as refresh-opportunity proxy
y = (df["trend_direction"] == "down").astype(int)

# Use the feature matrix already prepared
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Positive class:", y.sum())
print("Negative class:", (y == 0).sum())

Feature matrix shape: (30000, 27)
Target shape: (30000,)
Positive class: 16262
Negative class: 13738


In [17]:
!python scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship/outputs/refresh_queue.csv
Wrote model report: /content/flyr

In [18]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.6 MB/s eta 0:00:00


In [20]:
!python scripts/05_build_pdf_report.py

Wrote PDF report: /content/flyrank-ml-internship/outputs/flyrank_refresh_model_results.pdf


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Results

The models were evaluated using a client-level holdout split, with 27,675
records used for training and 2,325 records held out for evaluation.

The transparent baseline achieved an F1 score of 0.274 and a ROC-AUC of 0.627.
Its Precision@50 was 0.24.

Three supervised models were compared. Logistic Regression achieved an F1
score of 0.566 and ROC-AUC of 0.700. Decision Tree achieved an F1 score of
0.634 and ROC-AUC of 0.742.

Random Forest performed best on the ranking objective, achieving an F1 score
of 0.640, ROC-AUC of 0.750, and Precision@50 of 0.74.

The Random Forest was therefore selected as the final model because the
project focuses on ranking a small set of pages for human review. Its
Precision@50 means that 37 of the top 50 ranked pages in the held-out
evaluation set were in the observed decline class.

The model improves substantially over the transparent baseline for the
ranking task, increasing Precision@50 from 0.24 to 0.74.

These results indicate that the model can support prioritization of pages for
review. They do not establish that every predicted page requires a content
refresh or that the model predicts future Google rankings.

In [22]:
results_table = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "F1": [
        0.2743,
        0.5662,
        0.6339,
        0.6395
    ],
    "ROC-AUC": [
        0.6269,
        0.7003,
        0.7415,
        0.7500
    ],
    "Precision@50": [
        0.24,
        0.40,
        0.58,
        0.74
    ]
})

display(results_table)

,Model,F1,ROC-AUC,Precision@50
0,Baseline,0.2743,0.6269,0.24
1,Logistic Regression,0.5662,0.7003,0.40
2,Decision Tree,0.6339,0.7415,0.58
3,Random Forest,0.6395,0.7500,0.74


## 5. Limitations

*What this work cannot claim.*

### Limitations

This analysis has several limitations.

First, the dataset does not contain a human-labelled outcome indicating that
a page actually required a content refresh. The target is therefore a proxy
based on observed performance decline.

Second, the results are based on the available anonymized dataset and its
observed signals. They should not be interpreted as causal evidence that a
specific content change will improve performance.

Third, the model is designed to prioritize pages for review rather than make
an automatic refresh decision. Human review is still required before taking
action.

Finally, the analysis does not establish or predict future Google rankings.
The results describe patterns measured in the available data and should be
treated as directional decision-support evidence.

### Ranked Recommendations

The final refresh queue ranks all 30,000 content items using the selected
Random Forest model. Pages are ordered by `final_refresh_score`, which combines
the model signal with the baseline refresh signal.

The highest-ranked pages should be treated as candidates for human review,
rather than as pages that are automatically confirmed to require a refresh.

The top-ranked pages show a common pattern of observed performance decline
combined with search visibility and content-quality signals. For example, the
highest-ranked records have `trend_direction = down`, high model probabilities,
and reason codes such as declining performance with demand and low CTR on
visible pages.

The `suggested_action` field provides an additional prioritization signal.
Depending on the observed signals, recommendations include reviewing and
refreshing the page, or reviewing specific aspects such as CTR or engagement.

This ranked queue can help a content team focus limited review effort on the
highest-priority pages instead of manually examining all 30,000 records.

In [24]:
# Show the highest-priority recommendations

top_recommendations = queue[
    [
        "final_rank",
        "content_id",
        "final_refresh_score",
        "best_model_probability",
        "confidence",
        "suggested_action",
        "final_reason_codes",
        "trend_direction",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update"
    ]
].head(20)

display(top_recommendations)

,final_rank,content_id,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes,trend_direction,avg_position,ctr,content_age_days,days_since_last_update
0,1,content_1f080331fa2b,81.734212,0.783472,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,down,6.8,0.05,165,104
1,2,content_d6570c51c9bd,81.603243,0.849842,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,10.1,0.00,165,104
2,3,content_6aa43079fb0c,81.544618,0.789490,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,3.8,0.07,139,104
3,4,content_72e800a9c214,81.169731,0.776297,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,8.2,0.12,139,104
4,5,content_e04eb9549989,80.957565,0.816010,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,3.6,0.09,131,104
5,6,content_b69288c5e701,80.798090,0.796332,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,6.4,0.07,144,104
6,7,content_9b6df29f7889,80.650656,0.846499,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,3.1,0.12,139,104
7,8,content_ba6f9dfcbca1,80.432641,0.827496,medium,refresh,declining_with_demand|model_decline_risk|visib...,down,20.1,0.09,124,104
8,9,content_4d76cdb3387b,80.428403,0.844030,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,down,2.7,0.13,131,104
9,10,content_b4f35d640b1c,80.428243,0.845325,medium,refresh,declining_with_demand|model_decline_risk|visib...,down,27.5,0.05,131,104


In [25]:
# Summary of recommended actions in the top 50 pages

top_50 = queue.head(50)

print("Top 50 recommendations:")
display(
    top_50["suggested_action"]
    .value_counts()
    .rename_axis("Suggested Action")
    .reset_index(name="Count")
)

Top 50 recommendations:


,Suggested Action,Count
0,refresh_and_review_ctr,37
1,refresh_and_review_engagement,8
2,refresh,5


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Acknowledgments

This project was completed as part of the FlyRank ML Internship.

I acknowledge the FlyRank internship team and the provided starter dataset,
documentation, and project structure, which supported the development of this
content refresh prioritization workflow.

The analysis, feature selection, model evaluation, and recommendations in this
notebook are intended for learning and decision-support purposes. The results
should not be interpreted as confirmation that a page must be refreshed or as
a prediction of future search-engine rankings.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Artifacts the Paper Embeds

The analysis produces charts and tables that summarize model performance,
ranking quality, and the main signals used for refresh prioritization.

The embedded artifacts are intended to make the results reproducible and easy
to interpret without exposing client-identifying information.

The main artifacts include:

- Model comparison table covering F1, ROC-AUC, and Precision@50
- Model performance visualizations generated by the evaluation pipeline
- Feature-importance visualization for the selected Random Forest model
- Ranked refresh queue showing the highest-priority pages for human review

In [29]:
from IPython.display import HTML, display
from pathlib import Path

chart_dir = Path("outputs/charts")

for chart in [
    "top_feature_importance.svg",
    "top_reason_codes.svg",
    "action_mix.svg"
]:
    print(chart)
    display(HTML(
        (chart_dir / chart).read_text()
    ))

top_feature_importance.svg


top_reason_codes.svg


action_mix.svg


### Artifact Interpretation

The feature-importance chart summarizes the strongest signals used by the
selected Random Forest model. The reason-code chart shows the main observed
conditions associated with pages entering the prioritized queue. The action
mix summarizes the types of review actions suggested by the pipeline.

Together, these artifacts provide both model-level evidence and
decision-support context for the ranked refresh queue.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
